<a href="https://colab.research.google.com/github/AizenSosuke2025/RecommendationSystem/blob/main/RecommendationModels2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Loading data...")
ratings_df = pd.read_csv('/content/drive/MyDrive/NetflixPrizeDataset/Copy of netflix_processed.csv',
                         dtype={
                             'MovieID': np.int16,
                             'CustomerID': np.int32,
                             'Rating': np.int8
                         })

print(f"Original dataset size: {len(ratings_df)} ratings")

# Count ratings per movie and per user
movie_counts = ratings_df['MovieID'].value_counts()
user_counts = ratings_df['CustomerID'].value_counts()

MIN_MOVIE_RATINGS = 500
MIN_USER_RATINGS = 50

# Get lists of popular movies and active users
popular_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
active_users = user_counts[user_counts >= MIN_USER_RATINGS].index

# Filter the dataframe to only include these users and movies
print("Filtering for active users and popular movies...")
subset_df = ratings_df[
    ratings_df['MovieID'].isin(popular_movies) &
    ratings_df['CustomerID'].isin(active_users)
].copy()

print(f"Reduced dataset size: {len(subset_df)} ratings")

# Re-map the IDs to new 0-based indices for the smaller matrix
subset_df['user_idx'] = subset_df['CustomerID'].astype('category').cat.codes
subset_df['movie_idx'] = subset_df['MovieID'].astype('category').cat.codes

num_users_subset = subset_df['user_idx'].nunique()
num_movies_subset = subset_df['movie_idx'].nunique()

print(f"New Unique Users: {num_users_subset}")
print(f"New Unique Movies: {num_movies_subset}")

## Item based Collaborative Filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

print("1. Splitting subset data...")
test_df = subset_df.groupby('user_idx').sample(frac=0.2, random_state=42)
train_df = subset_df.drop(test_df.index)

print("2. Building subset sparse matrices...")
train_matrix = csr_matrix(
    (train_df['Rating'], (train_df['user_idx'], train_df['movie_idx'])),
    shape=(num_users_subset, num_movies_subset)
)
test_matrix = csr_matrix(
    (test_df['Rating'], (test_df['user_idx'], test_df['movie_idx'])),
    shape=(num_users_subset, num_movies_subset)
)

print(f"Train matrix shape: {train_matrix.shape}")

print("3. Calculating Item-Item Cosine Similarity...")
# Transpose to (movies x users) to compare movies against each other
movie_user_matrix = train_matrix.T

# Calculate similarity
item_similarity = cosine_similarity(movie_user_matrix, dense_output=False)

print(f"Item similarity matrix shape: {item_similarity.shape}")
print("Done! The brain is built.")

In [ ]:
import numpy as np

def predict_rating(user_idx, movie_idx, train_matrix, item_similarity):
    movie_similarities = item_similarity[movie_idx].toarray().flatten()
    user_ratings = train_matrix[user_idx].toarray().flatten()

    # We only want to base our prediction on movies they've actually seen
    rated_indices = np.where(user_ratings > 0)[0]

    if len(rated_indices) == 0:
        return 0.0 # Cold start fallback

    # Extract the relevant similarities and ratings
    relevant_similarities = movie_similarities[rated_indices]
    relevant_ratings = user_ratings[rated_indices]

    numerator = np.dot(relevant_similarities, relevant_ratings)
    denominator = np.sum(np.abs(relevant_similarities))

    if denominator == 0:
        return 0.0 # Fallback (No similarity found)

    predicted_rating = numerator / denominator
    return predicted_rating

In [ ]:
def generate_top_10(user_idx, train_matrix, item_similarity):
    user_ratings = train_matrix[user_idx].toarray().flatten()
    unrated_movie_indices = np.where(user_ratings == 0)[0]

    predictions = []
    for movie_idx in unrated_movie_indices:
        pred = predict_rating(user_idx, movie_idx, train_matrix, item_similarity)
        predictions.append((movie_idx, pred))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_10_indices = [x[0] for x in predictions[:10]]
    return top_10_indices

In [ ]:
import numpy as np
import pandas as pd

print("Starting final evaluation...")
actual_ratings = []
predicted_ratings = []
total_ap_10 = 0.0
user_count_map = 0

all_test_users = test_df['user_idx'].unique()
sample_size = min(500, len(all_test_users))

# Using a fixed seed for result reproducibility
np.random.seed(42)
unique_test_users = np.random.choice(all_test_users, size=sample_size, replace=False)

print(f"Evaluating across a fast sample of {len(unique_test_users)} test users...")

for count, user_idx in enumerate(unique_test_users):
    # PREDICT RATINGS FOR RMSE
    user_test_interactions = test_df[test_df['user_idx'] == user_idx]

    for _, row in user_test_interactions.iterrows():
        movie_idx = int(row['movie_idx'])
        actual_r = row['Rating']
        pred_r = predict_rating(user_idx, movie_idx, train_matrix, item_similarity)
        if pred_r > 0:
            actual_ratings.append(actual_r)
            predicted_ratings.append(pred_r)

    # 2. GENERATE TOP-10 & EVALUATE MAP@10
    # Define actual ground-truth relevant items (Rating >= 3.5 per instructions)
    actual_relevant_movies = set(user_test_interactions[user_test_interactions['Rating'] >= 3.5]['movie_idx'])

    if len(actual_relevant_movies) > 0:
        top_10_recommendations = generate_top_10(user_idx, train_matrix, item_similarity)
        num_hits = 0
        sum_precisions = 0.0

        for k, movie_idx in enumerate(top_10_recommendations):
            if movie_idx in actual_relevant_movies:
                num_hits += 1
                precision_at_k = num_hits / (k + 1)
                sum_precisions += precision_at_k

        ap_10 = sum_precisions / min(10, len(actual_relevant_movies))
        total_ap_10 += ap_10
        user_count_map += 1

    # Progress tracker
    if (count + 1) % 100 == 0:
        print(f"Processed {count + 1}/{len(unique_test_users)} users...")

# Compute final RMSE
actual_ratings = np.array(actual_ratings)
predicted_ratings = np.array(predicted_ratings)
rmse_score = np.sqrt(np.mean((actual_ratings - predicted_ratings) ** 2))

# Compute final MAP@10
final_map_10 = total_ap_10 / user_count_map if user_count_map > 0 else 0.0

print("\n================ EVALUATION RESULTS ================")
print(f"Final RMSE:    {rmse_score:.4f}")
print(f"Final MAP@10:  {final_map_10:.4f}")
print("====================================================")

## User based Collaborative Filtering


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Calculating User-User Cosine Similarity...")

# We do not transpose the matrix this time, we are comparing row to row (user to user).
user_similarity = cosine_similarity(train_matrix, dense_output=False)

print(f"User similarity matrix shape: {user_similarity.shape}")

In [ ]:
def predict_user_based(user_idx, movie_idx, train_matrix, user_similarity):
    user_similarities = user_similarity[user_idx].toarray().flatten()
    movie_ratings = train_matrix[:, movie_idx].toarray().flatten()
    rated_indices = np.where(movie_ratings > 0)[0]
    rated_indices = rated_indices[rated_indices != user_idx]

    if len(rated_indices) == 0:
        return 0.0 # Cold start fallback
    relevant_similarities = user_similarities[rated_indices]
    relevant_ratings = movie_ratings[rated_indices]

    numerator = np.dot(relevant_similarities, relevant_ratings)
    denominator = np.sum(np.abs(relevant_similarities))

    if denominator == 0:
        return 0.0

    return numerator / denominator

In [ ]:
def generate_top_10_user_based(user_idx, train_matrix, user_similarity):
    user_ratings = train_matrix[user_idx].toarray().flatten()
    unrated_movie_indices = np.where(user_ratings == 0)[0]

    predictions = []

    for movie_idx in unrated_movie_indices:
        pred = predict_user_based(user_idx, movie_idx, train_matrix, user_similarity)
        predictions.append((movie_idx, pred))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_10_indices = [x[0] for x in predictions[:10]]

    return top_10_indices

In [ ]:
import numpy as np
import pandas as pd

print("Starting final User-Based evaluation...")

actual_ratings_ub = []
predicted_ratings_ub = []

total_ap_10_ub = 0.0
user_count_map_ub = 0

all_test_users = test_df['user_idx'].unique()
sample_size = min(100, len(all_test_users))
np.random.seed(42)  # Same seed as before!
unique_test_users = np.random.choice(all_test_users, size=sample_size, replace=False)

print(f"Evaluating across a fast sample of {len(unique_test_users)} test users...")

for count, user_idx in enumerate(unique_test_users):
    # PREDICT RATINGS FOR RMSE
    user_test_interactions = test_df[test_df['user_idx'] == user_idx]

    for _, row in user_test_interactions.iterrows():
        movie_idx = int(row['movie_idx'])
        actual_r = row['Rating']

        pred_r = predict_user_based(user_idx, movie_idx, train_matrix, user_similarity)

        if pred_r > 0:
            actual_ratings_ub.append(actual_r)
            predicted_ratings_ub.append(pred_r)

    # GENERATE TOP-10 & EVALUATE MAP@10
    actual_relevant_movies = set(user_test_interactions[user_test_interactions['Rating'] >= 3.5]['movie_idx'])

    if len(actual_relevant_movies) > 0:
        top_10_recommendations = generate_top_10_user_based(user_idx, train_matrix, user_similarity)

        num_hits = 0
        sum_precisions = 0.0

        for k, movie_idx in enumerate(top_10_recommendations):
            if movie_idx in actual_relevant_movies:
                num_hits += 1
                precision_at_k = num_hits / (k + 1)
                sum_precisions += precision_at_k

        ap_10 = sum_precisions / min(10, len(actual_relevant_movies))
        total_ap_10_ub += ap_10
        user_count_map_ub += 1

    # Progress tracker
    if (count + 1) % 20 == 0:
        print(f"Processed {count + 1}/{len(unique_test_users)} users...")

actual_ratings_ub = np.array(actual_ratings_ub)
predicted_ratings_ub = np.array(predicted_ratings_ub)
rmse_score_ub = np.sqrt(np.mean((actual_ratings_ub - predicted_ratings_ub) ** 2))

final_map_10_ub = total_ap_10_ub / user_count_map_ub if user_count_map_ub > 0 else 0.0

print("\n================ USER-BASED EVALUATION RESULTS ================")
print(f"Final User-Based RMSE:    {rmse_score_ub:.4f}")
print(f"Final User-Based MAP@10:  {final_map_10_ub:.4f}")
print("===============================================================")

## Matrix Factorization / SVD

In [ ]:
from scipy.sparse.linalg import svds
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

print("1. Splitting subset data...")
test_df = subset_df.groupby('user_idx').sample(frac=0.2, random_state=42)
train_df = subset_df.drop(test_df.index)

print("2. Building subset sparse matrices...")
train_matrix = csr_matrix(
    (train_df['Rating'], (train_df['user_idx'], train_df['movie_idx'])),
    shape=(num_users_subset, num_movies_subset)
)
test_matrix = csr_matrix(
    (test_df['Rating'], (test_df['user_idx'], test_df['movie_idx'])),
    shape=(num_users_subset, num_movies_subset)
)

print(f"Train matrix shape: {train_matrix.shape}")

print("Starting Matrix Factorization (SVD)...")

# SVD requires the matrix to be in float format
train_matrix_float = train_matrix.astype(float)
k = 50
U, sigma, Vt = svds(train_matrix_float, k=k)
sigma_diag = np.diag(sigma)

print(f"User matrix (U) shape: {U.shape}")
print(f"Sigma matrix shape: {sigma_diag.shape}")
print(f"Item matrix (V^T) shape: {Vt.shape}")

In [ ]:
def predict_svd(user_idx, movie_idx, U, sigma_diag, Vt):
    user_vector = U[user_idx, :]
    movie_vector = Vt[:, movie_idx]

    prediction = np.dot(np.dot(user_vector, sigma_diag), movie_vector)

    # Cap predictions to the valid 1-5 star range
    if prediction < 1:
        return 1.0
    elif prediction > 5:
        return 5.0
    else:
        return prediction

In [ ]:
def generate_top_10_svd(user_idx, train_matrix, U, sigma_diag, Vt):
    user_ratings = train_matrix[user_idx].toarray().flatten()
    unrated_movie_indices = np.where(user_ratings == 0)[0]

    predictions = []

    for movie_idx in unrated_movie_indices:
        pred = predict_svd(user_idx, movie_idx, U, sigma_diag, Vt)
        predictions.append((movie_idx, pred))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_10_indices = [x[0] for x in predictions[:10]]

    return top_10_indices

In [ ]:
import numpy as np
import pandas as pd

print("Starting final SVD evaluation...")

actual_ratings_svd = []
predicted_ratings_svd = []

total_ap_10_svd = 0.0
user_count_map_svd = 0

all_test_users = test_df['user_idx'].unique()
sample_size = min(8000, len(all_test_users))
np.random.seed(42)  # Critical for fair comparison
unique_test_users = np.random.choice(all_test_users, size=sample_size, replace=False)

print(f"Evaluating across a fast sample of {len(unique_test_users)} test users...")

for count, user_idx in enumerate(unique_test_users):
    # PREDICT RATINGS FOR RMSE
    user_test_interactions = test_df[test_df['user_idx'] == user_idx]

    for _, row in user_test_interactions.iterrows():
        movie_idx = int(row['movie_idx'])
        actual_r = row['Rating']
        pred_r = predict_svd(user_idx, movie_idx, U, sigma_diag, Vt)

        if pred_r > 0:
            actual_ratings_svd.append(actual_r)
            predicted_ratings_svd.append(pred_r)

    # GENERATE TOP-10 & EVALUATE MAP@10
    actual_relevant_movies = set(user_test_interactions[user_test_interactions['Rating'] >= 3.5]['movie_idx'])

    if len(actual_relevant_movies) > 0:
        top_10_recommendations = generate_top_10_svd(user_idx, train_matrix, U, sigma_diag, Vt)

        num_hits = 0
        sum_precisions = 0.0

        for k_idx, movie_idx in enumerate(top_10_recommendations):
            if movie_idx in actual_relevant_movies:
                num_hits += 1
                precision_at_k = num_hits / (k_idx + 1)
                sum_precisions += precision_at_k

        ap_10 = sum_precisions / min(10, len(actual_relevant_movies))
        total_ap_10_svd += ap_10
        user_count_map_svd += 1

    # Progress tracker
    if (count + 1) % 500 == 0:
        print(f"Processed {count + 1}/{len(unique_test_users)} users...")

actual_ratings_svd = np.array(actual_ratings_svd)
predicted_ratings_svd = np.array(predicted_ratings_svd)
rmse_score_svd = np.sqrt(np.mean((actual_ratings_svd - predicted_ratings_svd) ** 2))

final_map_10_svd = total_ap_10_svd / user_count_map_svd if user_count_map_svd > 0 else 0.0

print("\n================ SVD EVALUATION RESULTS ================")
print(f"Final SVD RMSE:    {rmse_score_svd:.4f}")
print(f"Final SVD MAP@10:  {final_map_10_svd:.4f}")
print("========================================================")

## Success and failures


In [ ]:
import numpy as np
movies_df = pd.read_csv('/content/drive/MyDrive/NetflixPrizeDataset/movie_titles.csv',
                        header=None,
                        names=['MovieID', 'Year', 'Title'],
                        encoding='iso-8859-1',
                        on_bad_lines='skip')

print("Setting up Movie Title translator...")

# Create a mapping dictionary for our subset indices back to the original MovieIDs (We use the categories from the subset we created in Step 3).
subset_movie_mapping = dict(enumerate(subset_df['MovieID'].astype('category').cat.categories))

def get_movie_info(movie_idx):
    """Translates a matrix index into a real Movie Title and Year."""
    real_id = subset_movie_mapping[movie_idx]
    movie_row = movies_df[movies_df['MovieID'] == real_id]

    if not movie_row.empty:
        title = movie_row['Title'].values[0]
        year = movie_row['Year'].values[0]
        return f"{title} ({year})"
    return "Unknown Movie"

sample_user_idx = unique_test_users[0] # Change this index to look at different users

print(f"\nAnalyzing User Index: {sample_user_idx}")
print("-" * 50)
user_test_data = test_df[(test_df['user_idx'] == sample_user_idx) & (test_df['Rating'] >= 4)]
print("MOVIES THE USER ACTUALLY LOVED (From Test Set):")
for _, row in user_test_data.iterrows():
    print(f"⭐ {row['Rating']} Stars | {get_movie_info(int(row['movie_idx']))}")

print("\n" + "-" * 50)

# Generate the Top-10 Recommendations using SVD
print("TOP 10 SVD RECOMMENDATIONS:")
top_10_indices = generate_top_10_svd(sample_user_idx, train_matrix, U, sigma_diag, Vt)

for rank, movie_idx in enumerate(top_10_indices):
    title = get_movie_info(movie_idx)
    print(f"#{rank + 1}: {title}")

## ALS

In [ ]:
!pip install implicit

In [ ]:
import implicit
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# print("1. Splitting subset data...")
# test_df = subset_df.groupby('user_idx').sample(frac=0.2, random_state=42)
# train_df = subset_df.drop(test_df.index)

# print("2. Building subset sparse matrices...")
# train_matrix = csr_matrix(
#     (train_df['Rating'], (train_df['user_idx'], train_df['movie_idx'])),
#     shape=(num_users_subset, num_movies_subset)
# )
# test_matrix = csr_matrix(
#     (test_df['Rating'], (test_df['user_idx'], test_df['movie_idx'])),
#     shape=(num_users_subset, num_movies_subset)
# )

print(f"Train matrix shape: {train_matrix.shape}")
print("Initializing Alternating Least Squares (ALS)...")
# Initialize the ALS model
als_model = implicit.als.AlternatingLeastSquares(
    factors=50,
    regularization=0.05,
    iterations=20,
    random_state=42
)

print("Training ALS Model (Watch how fast this is)...")
# Fit the model using the sparse matrix
als_model.fit(train_matrix)
print("Training Complete!")

In [ ]:
print("Starting ALS Evaluation...")

actual_ratings_als = []
predicted_ratings_als = []
total_ap_10_als = 0.0
user_count_map_als = 0

# Test on a fast sample of 1000 users
all_test_users = test_df['user_idx'].unique()
sample_size = min(1000, len(all_test_users))
np.random.seed(42)
unique_test_users = np.random.choice(all_test_users, size=sample_size, replace=False)

# Extract the raw NumPy factor arrays for exact RMSE prediction
user_factors = als_model.user_factors
item_factors = als_model.item_factors

for count, user_idx in enumerate(unique_test_users):
    user_test_interactions = test_df[test_df['user_idx'] == user_idx]

    # RMSE EVALUATION
    for _, row in user_test_interactions.iterrows():
        movie_idx = int(row['movie_idx'])
        actual_r = row['Rating']

        pred_r = np.dot(user_factors[user_idx], item_factors[movie_idx])
        pred_r = max(1.0, min(5.0, pred_r))

        actual_ratings_als.append(actual_r)
        predicted_ratings_als.append(pred_r)

    # MAP@10 EVALUATION
    actual_relevant_movies = set(user_test_interactions[user_test_interactions['Rating'] >= 3.5]['movie_idx'])

    if len(actual_relevant_movies) > 0:
        # filter_already_liked_items=True automatically removes the training data
        recommendations, _ = als_model.recommend(
            user_idx,
            train_matrix[user_idx],
            N=10,
            filter_already_liked_items=True
        )

        num_hits = 0
        sum_precisions = 0.0

        for k_idx, movie_idx in enumerate(recommendations):
            if movie_idx in actual_relevant_movies:
                num_hits += 1
                precision_at_k = num_hits / (k_idx + 1)
                sum_precisions += precision_at_k

        ap_10 = sum_precisions / min(10, len(actual_relevant_movies))
        total_ap_10_als += ap_10
        user_count_map_als += 1

# Calculate Final Metrics
rmse_score_als = np.sqrt(np.mean((np.array(actual_ratings_als) - np.array(predicted_ratings_als)) ** 2))
final_map_10_als = total_ap_10_als / user_count_map_als if user_count_map_als > 0 else 0.0

print("\n================ ALS EVALUATION RESULTS ================")
print(f"Final ALS RMSE:    {rmse_score_als:.4f}")
print(f"Final ALS MAP@10:  {final_map_10_als:.4f}")
print("========================================================")

In [ ]:
import numpy as np
movies_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/recommendation system/NetflixPrizeDataset/movie_titles.csv',
                        header=None,
                        names=['MovieID', 'Year', 'Title'],
                        encoding='iso-8859-1',
                        on_bad_lines='skip')

print("Setting up Movie Title translator...")
subset_movie_mapping = dict(enumerate(subset_df['MovieID'].astype('category').cat.categories))

def get_movie_info(movie_idx):
    """Translates a matrix index into a real Movie Title and Year."""
    real_id = subset_movie_mapping[movie_idx]
    movie_row = movies_df[movies_df['MovieID'] == real_id]

    if not movie_row.empty:
        title = movie_row['Title'].values[0]
        year = movie_row['Year'].values[0]
        return f"{title} ({year})"
    return "Unknown Movie"

sample_user_idx = unique_test_users[0]
print(f"\nAnalyzing ALS Recommendations for User Index: {sample_user_idx}")
print("-" * 50)

# The .recommend() function returns two arrays: the movie indices and their confidence scores
als_recommendations, als_scores = als_model.recommend(
    sample_user_idx,
    train_matrix[sample_user_idx],
    N=10,
    filter_already_liked_items=True
)

print("TOP 10 ALS RECOMMENDATIONS:")
for rank, (movie_idx, score) in enumerate(zip(als_recommendations, als_scores)):
    title = get_movie_info(movie_idx)
    # We print the Confidence Score to show how ALS thinks differently than SVD
    print(f"#{rank + 1}: {title} (Confidence: {score:.2f})")